In [18]:
from dotenv import load_dotenv
load_dotenv()
from IPython.display import Markdown, display
from rich import print

In [2]:
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langchain_core.output_parsers import StrOutputParser

# Create LLM

In [3]:
llm = init_chat_model(
    model="mistral-medium-latest",
    model_provider="mistralai",
)

# Create the tool

In [5]:
@tool
def wordcounter(sentence : str) -> int:
    """Counts the number of word in the sentence"""
    return (sentence.strip()).count(" ")+1

In [7]:
wordcounter.invoke("Hello how are you?") #This is now a tool so we can invoke it

4

In [27]:
print(wordcounter.description)
print(wordcounter.args)
print(wordcounter.name)
print(wordcounter)

Counts the number of word in the sentence

{'sentence': {'title': 'Sentence', 'type': 'string'}}

wordcounter

StructuredTool(
    name='wordcounter',
    description='Counts the number of word in the sentence',
    args_schema=<class 'langchain_core.utils.pydantic.wordcounter'>,
    func=<function wordcounter at 0x0000013DAF87E480>
)

# Binding Tool

In [13]:
llm_with_tool = llm.bind_tools([wordcounter])

In [25]:
print(llm_with_tool.invoke("Hello how are you?"))

AIMessage(
    content="Hello! I'm just a program, so I don't have feelings, but I'm here and ready to help you. How are 
you?",
    additional_kwargs={},
    response_metadata={
        'token_usage': {
            'prompt_tokens': 83,
            'total_tokens': 113,
            'completion_tokens': 30,
            'prompt_tokens_details': {'cached_tokens': 0}
        },
        'model_name': 'mistral-medium-latest',
        'model': 'mistral-medium-latest',
        'finish_reason': 'stop',
        'model_provider': 'mistralai'
    },
    id='lc_run--019f8b24-e3f4-75d1-a5e0-0cb4db06c01f-0',
    tool_calls=[],
    invalid_tool_calls=[],
    usage_metadata={'input_tokens': 83, 'output_tokens': 30, 'total_tokens': 113}
)

In [24]:
print(llm_with_tool.invoke("How many words are in: 'Hello how are you?'"))

AIMessage(
    content='',
    additional_kwargs={
        'tool_calls': [
            {
                'id': 'Xf2AvXt3Y',
                'type': 'function',
                'function': {'name': 'wordcounter', 'arguments': '{"sentence": "Hello how are you?"}'},
                'index': 0
            }
        ]
    },
    response_metadata={
        'token_usage': {
            'prompt_tokens': 90,
            'total_tokens': 106,
            'completion_tokens': 16,
            'prompt_tokens_details': {'cached_tokens': 0}
        },
        'model_name': 'mistral-medium-latest',
        'model': 'mistral-medium-latest',
        'finish_reason': 'tool_calls',
        'model_provider': 'mistralai'
    },
    id='lc_run--019f8b24-9ff5-7150-828a-bdd8f6c26443-0',
    tool_calls=[
        {
            'name': 'wordcounter',
            'args': {'sentence': 'Hello how are you?'},
            'id': 'Xf2AvXt3Y',
            'type': 'tool_call'
        }
    ],
    invalid_tool_calls=[],
    usage_metadata={'input_tokens': 90, 'output_tokens': 16, 'total_tokens': 106}
)

# Tool Calling and Execution

In [21]:
tools={
    "wordcounter": wordcounter
}

In [22]:
result = llm_with_tool.invoke("How many words are in: 'Hello how are you?'")

In [23]:
if result.tool_calls:
    tool_name=result.tool_calls[0]["name"]
    tool_message=tools[tool_name].invoke(result.tool_calls[0])
    print(tool_message)

ToolMessage(content='4', name='wordcounter', tool_call_id='FDvDiBqfh')

In [26]:
print(tool_message.content)

4

## We can bind them with memory using HumanMessage, ToolMessage

In [28]:
from langchain.messages import HumanMessage, ToolMessage

In [35]:
message=[]

In [31]:
prompt = "What is the number of words inside 'I don't like to play chess because it's boring piece of trash ahahahaha. Lol Lol there should be no reason to play at all."

In [36]:
message.append(HumanMessage(prompt))

In [37]:
message

[HumanMessage(content="What is the number of words inside 'I don't like to play chess because it's boring piece of trash ahahahaha. Lol Lol there should be no reason to play at all.", additional_kwargs={}, response_metadata={})]

In [38]:
llm_response=llm_with_tool.invoke(message)

In [39]:
message.append(llm_response)

In [ ]:
if llm_response.tool_calls:
    tool_used=llm_response.tool_calls[0]['name']
    tool_response=tools[tool_used].invoke(llm_response.tool_calls[0])
    message.append(tool_response)
    print(tool_response)

24

In [45]:
tool_response

ToolMessage(content='24', name='wordcounter', tool_call_id='1WLdPtxR0')

In [47]:
print(message)

[
    HumanMessage(
        content="What is the number of words inside 'I don't like to play chess because it's boring piece of trash 
ahahahaha. Lol Lol there should be no reason to play at all.",
        additional_kwargs={},
        response_metadata={}
    ),
    AIMessage(
        content='',
        additional_kwargs={
            'tool_calls': [
                {
                    'id': '1WLdPtxR0',
                    'type': 'function',
                    'function': {
                        'name': 'wordcounter',
                        'arguments': '{"sentence": "I don\'t like to play chess because it\'s boring piece of trash
ahahahaha. Lol Lol there should be no reason to play at all."}'
                    },
                    'index': 0
                }
            ]
        },
        response_metadata={
            'token_usage': {
                'prompt_tokens': 119,
                'total_tokens': 163,
                'completion_tokens': 44,
                'prompt_tokens_details': {'cached_tokens': 0}
            },
            'model_name': 'mistral-medium-latest',
            'model': 'mistral-medium-latest',
            'finish_reason': 'tool_calls',
            'model_provider': 'mistralai'
        },
        id='lc_run--019f8b38-567e-7633-827c-f97341761625-0',
        tool_calls=[
            {
                'name': 'wordcounter',
                'args': {
                    'sentence': "I don't like to play chess because it's boring piece of trash ahahahaha. Lol Lol 
there should be no reason to play at all."
                },
                'id': '1WLdPtxR0',
                'type': 'tool_call'
            }
        ],
        invalid_tool_calls=[],
        usage_metadata={'input_tokens': 119, 'output_tokens': 44, 'total_tokens': 163}
    ),
    ToolMessage(content='24', name='wordcounter', tool_call_id='1WLdPtxR0')
]

In [50]:
print(display(Markdown(llm_with_tool.invoke(message).content)))

The sentence **"I don't like to play chess because it's boring piece of trash ahahahaha. Lol Lol there should be no reason to play at all."** contains **24 words**.

None